# 02 — Audio Transcription: Input & Output

**Audience**: Developers who completed notebooks 00 and 01  
**Duration**: ~25 minutes  
**Goal**: Enable and use input/output audio transcription to capture spoken words as text in real time

---

## Introduction: Why Transcription?

The Gemini Live API supports two transcription channels that run **in parallel with audio**:

| Channel | What it transcribes | `resp` field |
|---|---|---|
| **Input transcription** | What the *user* said (audio you sent to Gemini) | `resp.server_content.input_transcription.text` |
| **Output transcription** | What *Gemini* is saying (its spoken response) | `resp.server_content.output_transcription.text` |

### Use cases

- **Accessibility** — display captions for users who are hard of hearing
- **Logging & audit trails** — record what was said in a voice session for compliance
- **Search & analytics** — index conversation text for later retrieval
- **Debugging** — verify Gemini understood your input correctly
- **Multi-modal UX** — show text alongside audio in voice interfaces

### How transcription arrives

Transcription text arrives **incrementally** as Gemini processes audio — you get partial words first, then corrections. This is similar to how speech-to-text services stream results.

```
resp.server_content.output_transcription.text  → "The"
resp.server_content.output_transcription.text  → "The Gemini"
resp.server_content.output_transcription.text  → "The Gemini Live"
resp.server_content.output_transcription.text  → "The Gemini Live API"
...
```

> **Important**: Transcription and audio arrive in *separate* response objects. You must collect both independently and they may interleave.

## Step 1: Install Dependencies

In [ ]:
!pip install google-genai nest_asyncio numpy --quiet

## Step 2: Setup & Imports

In [ ]:
import nest_asyncio; nest_asyncio.apply()

import asyncio
import io
import json
import os
import wave
from datetime import datetime

import numpy as np
import IPython.display as ipd

from google import genai
from google.genai import types

from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

API_KEY     = os.environ.get("GEMINI_API_KEY", "")
MODEL       = "gemini-3.1-flash-live-preview"
client      = genai.Client(api_key=API_KEY)
INPUT_RATE  = 16_000
OUTPUT_RATE = 24_000

print("✓ Setup complete")

## Utility Functions

In [ ]:
def make_pcm(text_hint: str = "", duration: float = 2.0, rate: int = 16000) -> bytes:
    """Generate a sine-wave tone as raw PCM16 bytes."""
    freq    = 220 if "low" in text_hint else 440
    t       = np.linspace(0, duration, int(rate * duration), endpoint=False)
    samples = (np.sin(2 * np.pi * freq * t) * 0.3 * 32767).astype(np.int16)
    return samples.tobytes()


def play_pcm(raw_bytes: bytes, rate: int = 24000) -> ipd.Audio:
    """Wrap raw PCM16 bytes in an IPython Audio widget."""
    arr = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    return ipd.Audio(arr, rate=rate, autoplay=False)


def make_spoken_pcm(text: str, duration: float = 3.0, rate: int = 16000) -> bytes:
    """
    Generate a multi-tone PCM clip that 'sounds like' spoken audio.
    Uses a mix of frequencies in the speech range (100–3000 Hz) to simulate
    a voice formant pattern. Not real speech, but richer than a single tone.

    In a real app, you would capture this from a microphone.
    """
    t = np.linspace(0, duration, int(rate * duration), endpoint=False)

    # Fundamental + first three harmonics, mimicking a vowel sound
    signal  = np.sin(2 * np.pi * 120 * t) * 0.40   # fundamental (120 Hz)
    signal += np.sin(2 * np.pi * 240 * t) * 0.20   # 2nd harmonic
    signal += np.sin(2 * np.pi * 480 * t) * 0.15   # 3rd harmonic
    signal += np.sin(2 * np.pi * 800 * t) * 0.10   # F1 formant region
    signal += np.sin(2 * np.pi * 1800 * t) * 0.08  # F2 formant region

    # Amplitude envelope: soft attack, sustain, release
    fade = int(rate * 0.05)
    env  = np.ones_like(signal)
    env[:fade]  = np.linspace(0, 1, fade)
    env[-fade:] = np.linspace(1, 0, fade)
    signal *= env

    return (signal * 0.5 * 32767).astype(np.int16).tobytes()


def pcm_to_wav_bytes(pcm: bytes, rate: int = 16000) -> bytes:
    """Wrap PCM16 bytes in a WAV container."""
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(rate)
        wf.writeframes(pcm)
    return buf.getvalue()


print("✓ Helpers defined: make_pcm, play_pcm, make_spoken_pcm, pcm_to_wav_bytes")

## The Transcription Config

Enabling transcription is a one-line addition to your `LiveConnectConfig`. You can enable either or both channels independently.

In [ ]:
# ── Full transcription config (both input and output) ─────────────────────────
TRANSCRIPTION_CONFIG = types.LiveConnectConfig(
    response_modalities=["AUDIO"],
    # Enable transcription of what the user sends (input audio)
    input_audio_transcription=types.AudioTranscriptionConfig(),
    # Enable transcription of what Gemini says (output audio)
    output_audio_transcription=types.AudioTranscriptionConfig(),
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
        )
    ),
)

# ── Output-only transcription (most common for logging Gemini's speech) ────────
OUTPUT_TRANSCRIPTION_ONLY = types.LiveConnectConfig(
    response_modalities=["AUDIO"],
    output_audio_transcription=types.AudioTranscriptionConfig(),
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
        )
    ),
)

# ── Note: TEXT modality is NOT supported — use AUDIO + output_audio_transcription
# TEXT_WITH_TRANSCRIPTION is removed; for text output use output_audio_transcription

print("Config objects created:")
print("  TRANSCRIPTION_CONFIG       — both input + output transcription")
print("  OUTPUT_TRANSCRIPTION_ONLY  — only transcribe Gemini's speech")


---
## Demo 1: Text Input + Output Transcription (Streaming Word-by-Word)

We send a text prompt and ask Gemini to respond in audio. With `output_audio_transcription` enabled, Gemini will stream the transcript of its own speech back to us in real time.

Watch how the transcript builds incrementally — this is the basis for live captions.

In [ ]:
async def demo_output_transcription() -> dict:
    """
    Send a text prompt, receive audio + streaming output transcription.
    Returns dict with 'audio' bytes and 'transcript' string.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        # Request transcription of Gemini's spoken response
        output_audio_transcription=types.AudioTranscriptionConfig(),
    )

    audio_chunks   = []
    transcript_parts = []

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("[Session open]")
        print("─" * 60)
        print("OUTPUT TRANSCRIPT (streaming):")
        print("─" * 60)

        # Send a text prompt — Gemini will respond in speech
        await session.send_realtime_input(
            text=(
                "Please say this exact text slowly and clearly: "
                "'Welcome to the Gemini Live API demonstration. "
                "I can speak and you can read what I say at the same time. "
                "This is output audio transcription.'"
            )
        )

        async for resp in session.receive():
            # ── Collect audio ──────────────────────────────────────────────────
            if resp.data:
                audio_chunks.append(resp.data)

            # ── Stream output transcription ────────────────────────────────────
            sc = resp.server_content
            if sc:
                # output_transcription arrives while Gemini is speaking
                if sc.output_transcription and sc.output_transcription.text:
                    text = sc.output_transcription.text
                    transcript_parts.append(text)
                    # Print incrementally — no newline, so we see it build
                    print(text, end="", flush=True)

                if sc.turn_complete:
                    print()  # final newline
                    print("─" * 60)
                    print("[turn complete]")
                    break

            if resp.go_away:
                print("\n[go_away]")
                break

    print("[Session closed]")
    return {
        "audio": b"".join(audio_chunks),
        "transcript": "".join(transcript_parts),
    }


print("── Demo 1: Text In → Audio + Output Transcription ──\n")
demo1_result = asyncio.run(demo_output_transcription())

print(f"\nAudio received : {len(demo1_result['audio']):,} bytes")
print(f"Transcript len : {len(demo1_result['transcript'])} characters")
print("\nFull transcript:")
print(f'  "{demo1_result["transcript"]}"')

In [ ]:
# Play back what Gemini said
if demo1_result["audio"]:
    dur = (len(demo1_result["audio"]) // 2) / OUTPUT_RATE
    print(f"Audio duration: {dur:.2f}s — listen and read the transcript above:")
    play_pcm(demo1_result["audio"], rate=OUTPUT_RATE)
else:
    print("No audio received")

---
## Demo 2: Audio Input + Input Transcription

Now we flip it around: we send synthetic audio **to** Gemini and ask Gemini to transcribe what it hears. This demonstrates `input_audio_transcription`.

Since our synthetic audio is not real speech, Gemini will either:
- Describe what it heard (a tone, silence, etc.)
- Return an empty or short transcription (no words recognised)

In a production app you would send real microphone audio and see the spoken words appear here.

In [ ]:
# Generate a spoken-style PCM clip (richer than a pure sine wave)
spoken_pcm = make_spoken_pcm(text="test phrase", duration=3.0, rate=INPUT_RATE)

print(f"Generated speech-like audio:")
print(f"  Duration : 3.0s")
print(f"  Bytes    : {len(spoken_pcm):,}")
print(f"  Format   : PCM16 @ {INPUT_RATE} Hz")
print("\nPreview (multi-formant tone):")
ipd.display(play_pcm(spoken_pcm, rate=INPUT_RATE))

In [ ]:
async def demo_input_transcription(pcm_bytes: bytes) -> dict:
    """
    Send audio to Gemini with input transcription enabled.
    Returns dict with 'audio' response bytes, 'input_transcript', 'output_transcript'.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        # Transcribe what the USER sends to Gemini
        input_audio_transcription=types.AudioTranscriptionConfig(),
        # Also transcribe Gemini's response
        output_audio_transcription=types.AudioTranscriptionConfig(),
        system_instruction=types.Content(parts=[types.Part(text=(
            "Listen carefully to the audio. "
            "Describe what you hear and respond conversationally. "
            "If you hear a tone rather than speech, say: "
            "'I heard a tone, not speech. In a real app, you would send microphone audio here.'"
        ))]),
    )

    audio_chunks   = []
    in_transcript  = []
    out_transcript = []

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("[Session open — audio input with transcription]")

        # Send the audio blob
        await session.send_realtime_input(
            audio=types.Blob(data=pcm_bytes, mime_type="audio/pcm;rate=16000")
        )
        print("[Audio sent]")

        print("─" * 50)

        async for resp in session.receive():
            if resp.data:
                audio_chunks.append(resp.data)

            sc = resp.server_content
            if sc:
                # ── Input transcription (what Gemini heard from us) ────────────
                if sc.input_transcription and sc.input_transcription.text:
                    text = sc.input_transcription.text
                    in_transcript.append(text)
                    print(f"[INPUT  TRANSCRIPT] {text}", flush=True)

                # ── Output transcription (what Gemini is saying) ───────────────
                if sc.output_transcription and sc.output_transcription.text:
                    text = sc.output_transcription.text
                    out_transcript.append(text)
                    print(f"[OUTPUT TRANSCRIPT] {text}", flush=True)

                if sc.turn_complete:
                    print("─" * 50)
                    print("[turn complete]")
                    break

            if resp.go_away:
                break

    print("[Session closed]")
    return {
        "audio":            b"".join(audio_chunks),
        "input_transcript":  "".join(in_transcript),
        "output_transcript": "".join(out_transcript),
    }


print("── Demo 2: Audio In + Input Transcription ──\n")
demo2_result = asyncio.run(demo_input_transcription(spoken_pcm))

print(f"\nInput transcript  : '{demo2_result['input_transcript']}'")
print(f"Output transcript : '{demo2_result['output_transcript'][:80]}...'")
print("\nGemini's audio response:")
play_pcm(demo2_result["audio"], rate=OUTPUT_RATE)

---
## Demo 3: Full-Duplex Transcript Logging — Multi-Turn Conversation

We run **multiple turns** in one session and collect a structured conversation log — both what the user said and what Gemini said, with timestamps.

This is the foundation for:
- Call centre transcription
- Meeting note generation
- Voice session replay
- Compliance recording

In [ ]:
async def demo_conversation_log() -> list[dict]:
    """
    Multi-turn session: send 3 questions as text, collect both the audio responses
    and output transcriptions into a structured conversation log.
    Returns a list of turn dicts: {turn, user_text, gemini_transcript, audio_bytes, timestamp}
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        system_instruction=types.Content(parts=[types.Part(text=(
            "You are a helpful assistant. Give short, clear answers (1–2 sentences). "
            "Be conversational."
        ))]),
    )

    # Our simulated user turns
    user_turns = [
        "What is the speed of light in metres per second?",
        "How many planets are in our solar system?",
        "What is the chemical symbol for gold?",
    ]

    conversation_log = []

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("[Session open — multi-turn transcript logging]\n")

        for turn_num, user_text in enumerate(user_turns, 1):
            turn_start = datetime.utcnow().isoformat() + "Z"

            print(f"╔══ Turn {turn_num} ══════════════════════════════════════════╗")
            print(f"║ USER    : {user_text}")
            print(f"║ TIME    : {turn_start}")

            # Send this turn's text
            await session.send_realtime_input(text=user_text)

            # Collect response for this turn
            turn_audio       = []
            turn_transcript  = []

            async for resp in session.receive():
                if resp.data:
                    turn_audio.append(resp.data)

                sc = resp.server_content
                if sc:
                    if sc.output_transcription and sc.output_transcription.text:
                        t = sc.output_transcription.text
                        turn_transcript.append(t)
                        print(f"║ GEMINI  : {t}", end="", flush=True)

                    if sc.turn_complete:
                        print()   # close the streaming line
                        break

                if resp.go_away:
                    break

            # Build this turn's log entry
            full_audio      = b"".join(turn_audio)
            full_transcript = "".join(turn_transcript).strip()
            audio_duration  = (len(full_audio) // 2) / OUTPUT_RATE

            print(f"║ AUDIO   : {len(full_audio):,} bytes ({audio_duration:.2f}s)")
            print(f"╚═══════════════════════════════════════════════════════╝\n")

            conversation_log.append({
                "turn":              turn_num,
                "timestamp":         turn_start,
                "user_text":         user_text,
                "gemini_transcript": full_transcript,
                "audio_bytes":       len(full_audio),
                "audio_duration_s":  round(audio_duration, 3),
                # Store audio separately — not serialisable to JSON
                "_audio": full_audio,
            })

    print("[Session closed]")
    return conversation_log


print("── Demo 3: Multi-Turn Conversation Logging ──\n")
conv_log = asyncio.run(demo_conversation_log())
print(f"\nCaptured {len(conv_log)} turns.")

In [ ]:
# ── Pretty-print the conversation log ──────────────────────────────────────────
print("═" * 60)
print(" CONVERSATION LOG")
print("═" * 60)

for entry in conv_log:
    print(f"\n  Turn {entry['turn']} — {entry['timestamp']}")
    print(f"  User   : {entry['user_text']}")
    print(f"  Gemini : {entry['gemini_transcript']}")
    print(f"  Audio  : {entry['audio_bytes']:,} bytes, {entry['audio_duration_s']}s")

print("\n" + "═" * 60)

In [ ]:
# ── Play each turn's audio ─────────────────────────────────────────────────────
for entry in conv_log:
    audio = entry["_audio"]
    if audio:
        print(f"Turn {entry['turn']}: {entry['gemini_transcript'][:60]}...")
        ipd.display(play_pcm(audio, rate=OUTPUT_RATE))

---
## Saving Transcripts to JSON

Conversation logs should be persisted for later analysis. We save the transcript (without the raw audio bytes) to a JSON file.

In [ ]:
import json

def save_transcript(log: list[dict], path: str) -> None:
    """
    Save a conversation log to a JSON file.
    Excludes raw audio bytes (the '_audio' key) which are not JSON-serialisable.

    Args:
        log  : list of turn dicts from demo_conversation_log()
        path : file path to write (will be created or overwritten)
    """
    # Build a serialisable version — drop the _audio key
    serialisable = [
        {k: v for k, v in entry.items() if k != "_audio"}
        for entry in log
    ]

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "session_id": datetime.utcnow().strftime("%Y%m%d_%H%M%S"),
                "model":      MODEL,
                "turns":      len(serialisable),
                "log":        serialisable,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    print(f"Saved transcript to: {path}")
    print(f"  {len(serialisable)} turns, {sum(e['audio_duration_s'] for e in serialisable):.2f}s total audio")


# Save the log from Demo 3
TRANSCRIPT_PATH = "/tmp/gemini_transcript.json"
save_transcript(conv_log, TRANSCRIPT_PATH)

# Read and display it
with open(TRANSCRIPT_PATH) as f:
    saved = json.load(f)

print("\n── Saved JSON (pretty-printed) ──")
print(json.dumps(saved, indent=2))

---
## Transcript Patterns Reference

Here is the complete receive-loop pattern showing every transcription field, annotated.

In [ ]:
# ── Reference: complete receive loop with all transcription fields ─────────────
#
# async for resp in session.receive():
#
#   # ── Raw audio output ──────────────────────────────────────────────────────
#   if resp.data:
#       audio_buffer.append(resp.data)          # PCM16 at 24000 Hz
#
#   # ── Server content (transcription, completion signals) ────────────────────
#   sc = resp.server_content
#   if sc:
#
#       # NOTE: response_modalities=["TEXT"] is NOT supported by this model.
#       # sc.model_turn / sc.model_turn.parts are only populated in TEXT mode.
#       # For text output, use output_audio_transcription instead (see below).
#
#       # Output transcription — what Gemini is *saying* right now
#       if sc.output_transcription and sc.output_transcription.text:
#           captions.append(sc.output_transcription.text)
#
#       # Input transcription — what Gemini *heard* from the user
#       if sc.input_transcription and sc.input_transcription.text:
#           user_log.append(sc.input_transcription.text)
#
#       # Turn complete signal — Gemini finished its response for this turn
#       if sc.turn_complete:
#           break
#
#   # ── Tool calls (see function calling notebooks) ───────────────────────────
#   if resp.tool_call:
#       for fc in resp.tool_call.function_calls:
#           # fc.name, fc.args, fc.id
#           pass
#
#   # ── Server shutting down — reconnect ─────────────────────────────────────
#   if resp.go_away:
#       break

print("Reference pattern loaded — read the comments above.")
print("No code to run in this cell.")


### Note: TEXT modality is not supported

The `gemini-3.1-flash-live-preview` model only supports `AUDIO` response modality.
Setting `response_modalities=["TEXT"]` causes a 1011 internal error on connect.

To get text from a Live API session, use **output audio transcription**:

```python
# ✅ Correct: AUDIO modality + transcription for text
config = types.LiveConnectConfig(
    response_modalities=["AUDIO"],
    output_audio_transcription=types.AudioTranscriptionConfig(),
)
...
async for resp in session.receive():
    if resp.server_content and resp.server_content.output_transcription:
        print(resp.server_content.output_transcription.text)  # transcript
    if resp.data:  # audio bytes
        audio_chunks.append(resp.data)
```

The convenience property `resp.text` maps to `model_turn.parts[].text` which
is only populated in TEXT modality. For audio sessions, always use
`resp.server_content.output_transcription.text` instead.


In [ ]:
async def demo_audio_transcription_shorthand() -> str:
    """
    Shows how to read transcription in AUDIO modality sessions.
    Use resp.server_content.output_transcription.text (not resp.text).
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
    )

    parts = []
    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(text="What year was Python created? One sentence.")

        async for resp in session.receive():
            # Transcript text from output_audio_transcription
            if resp.server_content and resp.server_content.output_transcription:
                if resp.server_content.output_transcription.text:
                    parts.append(resp.server_content.output_transcription.text)
                    print(resp.server_content.output_transcription.text, end="", flush=True)
            if resp.server_content and resp.server_content.turn_complete:
                print()
                break
            if resp.go_away:
                break

    return "".join(parts)


print("── Output transcription demo ──\n")
result = asyncio.run(demo_audio_transcription_shorthand())
print(f"\nCaptured: '{result.strip()}'")


---
## Key Takeaways

1. **Two transcription channels**: `input_audio_transcription` (user → Gemini) and `output_audio_transcription` (Gemini → user). Enable either or both via `LiveConnectConfig`.

2. **Transcription is incremental** — text arrives in chunks as Gemini processes audio. Append chunks to build the full transcript.

3. **Audio and transcript are separate** — `resp.data` carries audio bytes; `resp.server_content.output_transcription.text` carries transcript text. Both may arrive in the same receive loop iteration or different ones.

4. **Check `sc.turn_complete`** (not `resp.server_content.turn_complete`) to know when a full turn is done. Break the receive loop at this point before starting the next turn.

5. **`resp.text`** is a convenient shorthand for text-modality sessions, but use the explicit `sc.output_transcription.text` path for audio sessions.

6. **Save transcripts as JSON** — strip raw audio bytes before serialising; save audio separately as WAV files if needed.

7. **One session, many turns** — keep the session alive and send multiple `send_realtime_input` calls rather than reconnecting per turn. This is cheaper and faster.

---
**Series complete!** You have now covered:
- `00_live_api_basics.ipynb` — opening sessions, text/audio I/O, session lifecycle
- `01_audio_streaming.ipynb` — PCM format, WAV conversion, chunked mic streaming
- `02_transcription.ipynb` — input/output transcription, conversation logging, JSON export